(p1-theory-end2end-regression)=
# P1 이론: 머신러닝 프로젝트와 회귀 모델 훈련

**감사의 글**

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 강의노트이다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 권장한다. 자료를 공개한 오렐리앙 제롱과 일부 그림 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전한다.

**주요 내용**

1. **머신러닝 프로젝트와 데이터 분석**  
2. **데이터 준비에서 모델 평가까지**  
3. **회귀 모델은 무엇을 학습하는가**  

## 머신러닝 프로젝트와 데이터 분석

**문제 정의 → 데이터 이해 → train/test → EDA → 상관관계 → 특성 선택·공학 → 모델 훈련 → 모델 평가**

### 문제 정의와 데이터

머신러닝 프로젝트는 모델을 고르는 것보다 **무엇을 예측하려는지 명확히 하는 것**에서 시작한다.

여기서는 1990년 미국 캘리포니아의 20,640개 구역에 대한 데이터를 사용한다.
각 구역의 데이터는 
경도, 위도, 주택 건물 중위연령, 총 방 수, 총 침실 수, 인구, 가구 수, 중위소득, 중위 주택가격, 해안 근접도
등 총 10개의 특성이 포함되어 있다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/housing-data.png?raw=true" width="700">
</div>

캘리포니아는 미국 서부에 위치하며, 1990년 당시 2,976만명의 인구를 가졌다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/LA-USA01.png?raw=true" width="600">
</div>

이 프로젝트의 목표는 다른 특성을 이용하여 각 구역의 **중위 주택가격**을 예측하는 것이다.

- **특성**(feature): 중위 주택가격을 제외한 입력 정보
- **타깃**(target): 중위 주택가격
- **학습 유형**: 지도 학습
- **문제 유형**: 회귀

타깃이 **수치형 값**이므로 분류가 아니라 회귀 문제다.

또한 여러 특성을 이용해 하나의 수치형 타깃을 예측하므로 **다중 회귀**(multiple regression)이자 **단변량 회귀**(univariate regression) 문제다.

### 데이터셋 탐색

모델을 훈련하기 전에 데이터의 구조와 품질을 먼저 확인한다.

데이터셋 탐색에서는 다음을 살펴본다.

- 각 행과 열이 무엇을 의미하는가?
- 결측치는 있는가?
- 범주형 특성과 수치형 특성은 무엇인가?
- 특성의 값 범위와 분포는 어떠한가?
- 타깃 값에는 특별한 제한이나 이상한 점이 있는가?

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-05.png?raw=true" width="600">
</div>

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-05a.png?raw=true" width="350">
</div>

#### 범주형 특성과 수치형 특성

##### 범주형 특성

`ocean_proximity`는 해안과의 위치 관계를 나타내는 **범주형 특성**이다.
이 특성의 값은 `<1H OCEAN`, `INLAND`, `NEAR OCEAN`, `NEAR BAY`, `ISLAND`와 같은 **범주형 값**으로 구성된다.

| 특성값 | 설명 |
| :--- | :--- |
| <1H OCEAN | 해안에서 1시간 이내 |
| INLAND | 내륙 |
| NEAR OCEAN | 해안 근처 |
| NEAR BAY | 샌프란시스코의 Bay Area 구역 |
| ISLAND | 섬  |

##### 수치형 특성

나머지 특성은 수치형 특성이다.
수치형 특성에 대해서는 먼저 평균, 표준편차, 사분위수 등의 요약 통계를 살펴본다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/housing-describe.png?raw=true" width="100%">
</div>

히스토그램을 이용해 분포를 확인하는 것도 권장된다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/feature-histogram.png?raw=true" width="100%"></div></p>

히스토그램에서 몇 가지 중요한 사실을 확인할 수 있다.

- 특성마다 사용하는 단위와 스케일이 다르다.
- 일부 특성은 한쪽으로 치우친 분포를 보인다.
- 일부 특성은 값의 상한이 인위적으로 제한되어 있다.
- `total_bedrooms`에는 결측치가 존재한다.

이런 특징은 이후의 데이터 전처리 방법을 결정하는 근거가 된다.

### 훈련셋과 테스트셋

모델을 훈련하기 전에 전체 훈련 데이터를 **훈련셋**(training set)과 **테스트셋**(test set)으로 나눈다.

- **훈련셋**: 모델의 훈련과 모델 선택에 사용
- **테스트셋**: 최종적으로 선택된 모델의 일반화 성능을 평가하는 데 사용

> **테스트셋은 모델을 선택하거나 조정하는 과정에서 사용하지 않는다.**

#### 무작위 샘플링과 계층 샘플링

가장 간단한 방법은 전체 데이터에서 무작위로 샘플을 선택하는 **무작위 샘플링**(random sampling)이다.

그러나 데이터가 충분히 크지 않거나 중요한 집단의 비율을 유지해야 하는 경우에는 특정 집단이 훈련셋이나 테스트셋에 지나치게 많이 또는 적게 포함될 수 있다.

**계층 샘플링**(stratified sampling)은 중요한 특성을 기준으로 데이터를 여러 계층으로 나눈 뒤, 전체 데이터에서의 비율이 각 데이터셋에도 비슷하게 유지되도록 샘플을 선택하는 방법이다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-08.png?raw=true" width="400">
</div>

#### 중위소득 구간 활용

캘리포니아 주택 가격 데이터에서는 중위소득이 주택가격과 밀접하게 관련될 가능성이 있으므로, 
중위소득 구간을 5개로 구분한 다음에 중위소득 구간의 비율을 유지하도록 계층 샘플링을 사용할 수 있다.

| 구간 | 범위 |
| :---: | :--- |
| 1 | 0.0 ~ 1.5 |
| 2 | 1.5 ~ 3.0 |
| 3 | 3.0 ~ 4.5 |
| 4 | 4.5 ~ 6.0 |
| 5 | 6.0 ~  |

중요한 것은 특정 샘플링 방법을 항상 사용하는 것이 아니라,

> **훈련셋과 테스트셋이 실제 데이터의 중요한 특성을 적절히 대표하는가?**

를 확인하는 것이다.

### 훈련셋만 이용한 탐색적 데이터분석

훈련셋과 테스트셋을 나눈 뒤에는 **훈련셋만 이용해** 데이터의 관계를 탐색한다.

#### 지리적 정보 시각화

위도와 경도를 이용하면 주택가격의 지리적 분포를 확인할 수 있다.
점의 크기는 해당 구역의 인구를 나타내고, 점이 많이 모여 있을수록
지역의 인구 밀도가 높음을 의미한다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-09a.png?raw=true" width="600">
</div>

#### 상관관계

수치형 특성들 사이의 선형 관계는 **피어슨 상관계수**(Pearson correlation coefficient)로 확인할 수 있다.

상관계수는 -1에서 1 사이의 값을 가지며,

- 1에 가까울수록 강한 양의 선형 관계
- -1에 가까울수록 강한 음의 선형 관계
- 0에 가까울수록 약한 선형 관계

를 나타낸다.

상관계수가 0에 가깝다고 해서 두 특성 사이에 모든 종류의 관계가 없다는 뜻은 아니다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-12c.png?raw=true" width="100%">
</div>

중위 주택가격과 중위소득의 상관계수가 0.68로 상당히 높다.
이는 중위소득이 높을수록 중위 주택가격도 높아지는 선형적 경향이 비교적 강하게 나타남을 의미한다.
아래 산점도가 이 사실을 잘 확인시켜준다. 

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-13.png?raw=true" width="500"></div>

### 입력 데이터와 타깃 분리

지도 학습 모델을 훈련하려면 훈련셋을 **입력 데이터**와 **타깃**으로 분리한다.

입력 데이터에는 예측에 사용할 특성만 포함하고, 타깃에는 예측하려는 중위 주택가격만 포함한다.

이렇게 분리하면 이후의 전처리는 입력 데이터에 적용하고, 타깃은 모델이 예측해야 할 값으로 유지할 수 있다.

#### 특성 공학

기존 특성을 조합해 새로운 특성을 만들 수도 있다.

예를 들어 다음과 같은 비율은 원래 값보다 구역의 특성을 더 잘 나타낼 수 있다.

- 전체 방 수 대비 침실 수
- 가구당 방 수
- 가구당 평균 가구원 수

이처럼 기존 특성을 선택하거나 조합하여 모델에 더 유용한 특성을 만드는 과정을 **특성 공학**(feature engineering)이라고 한다.

## 데이터 준비에서 모델 평가까지

**전처리 → transformer → Pipeline/ColumnTransformer → 모델 훈련·비교 → 검증 → 최종 테스트**

### 데이터 정제와 전처리

실제 데이터는 그대로 모델에 넣기 어려운 경우가 많다.

이 데이터에서는 다음 전처리가 필요하다.

- 결측치 처리
- 범주형 특성의 원-핫 인코딩
- 수치형 특성의 스케일링
- 필요에 따른 특성 변환과 특성 조합

전처리의 목적은 단순히 데이터를 '깨끗하게' 만드는 것이 아니라,

> **모델이 의미 있게 사용할 수 있는 형태로 데이터를 변환하는 것**

이다.

#### 결측치 처리

`total_bedrooms`에는 일부 값이 누락되어 있는데,
많은 머신러닝 모델은 결측치가 있는 훈련셋을 활용하지 못한다.

결측치를 처리하는 대표적인 방법은

- 해당 샘플 제거
- 해당 특성 제거
- 평균, 중앙값 등의 대표값으로 대체

하는 것이다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/null-value01.png?raw=true" width="100%"></div></p>

#### 원-핫 인코딩

머신러닝 모델은 일반적으로 문자열 형태의 범주형 값을 직접 사용하지 못한다.

`ocean_proximity`와 같은 범주형 특성은 **원-핫 인코딩**(one-hot encoding)을 이용해 각 범주형 값을 0과 1로 표현된 새로운 특성으로 변환할 수 있다.

이렇게 새롭게 생성된 특성을 **더미**(dummy) 특성이라 한다.

`ocean_proximity` 특성은 다섯 개의 범주로 구성되기에 원-핫 인코딩을 통해 다섯 개의 새로운 더미 특성을 생성하여
원래 특성 대신 모델 훈련에 활용한다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/one_hot01.png?raw=true" width="700">
</div>

#### 특성 스케일링

수치형 특성마다 값의 범위가 크게 다르면 일부 머신러닝 알고리즘의 훈련에 영향을 줄 수 있다.

특성 스케일링은 정규화 또는 표준화를 통해 수치형 특성을 일정 크기로 변환하는 과정을 의미한다.

| 용어 | 정의 |
|------|------|
| **정규화**(Normalization) | 데이터 값을 일정한 범위로 맞추는 일반적인 과정.\n 대표적으로 특성값을 0과 1 사이의 값으로 변환하는 min-max 스케일링이 많이 활용됨 |
| **표준화**(Standardization) | 평균을 0, 표준편차를 1로 맞추는 과정 |


모든 모델이 스케일링에 똑같이 민감한 것은 아니지만, 여러 모델을 일관된 방식으로 비교할 때 유용하다.

#### 특성 변환

일부 수치형 특성은 한쪽으로 심하게 치우친 분포를 보인다.
이런 경우 로그 함수를 적용한 값으로 구성된 새로운 특성을 생성하는
로그 변환을 적용하여 분포의 치우침을 줄일 수 있다.

$$
\log(x)
$$

아래 그림은 구역별 인구로 구성된 `population` 특성값에 로그함수를 적용할 때 분포가 보다 균형잡히는 것을 잘 보여준다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/homl02-log_app.jpg?raw=true" width="500"></div></p>

### 사이킷런 모델, 변환기, 파이프라인

#### 모델과 변환기

사이킷런에서는 데이터 변환을 담당하는 객체와 예측을 담당하는 객체를 비슷한 방식으로 사용할 수 있다.

- **변환기**(transformer): `fit()`으로 필요한 정보를 결정하고 `transform()`으로 데이터를 변환
- **예측기**(predictor): `fit()`으로 모델을 훈련하고 `predict()`로 새로운 값을 예측

#### `Pipeline`과 `ColumnTransformer`

전처리 단계를 일일이 따로 실행하기보다 여러 단계를 하나의 **파이프라인**(pipeline)으로 묶는 것이 안전하고 재사용하기 쉽다.

`Pipeline`은 여러 처리 단계를 순서대로 연결한다.

반면에 `ColumnTransformer`는 서로 다른 특성 집합에 서로 다른 전처리를 적용한다.

예를 들어

- 수치형 특성 → 결측치 처리 + 스케일링
- 범주형 특성 → 결측치 처리 + 원-핫 인코딩

과 같이 서로 다른 처리를 하나의 전처리 과정으로 구성할 수 있다.

> **파이프라인의 핵심은 훈련 데이터에 적용한 전처리를 새로운 데이터에도 동일한 방식으로 적용하는 것**이다.

### 모델 선택

전처리가 완료되면 여러 회귀 모델을 같은 조건에서 훈련시켜 서로 비교할 수 있다.

이 장에서는 다음 세 회귀 모델을 비교한다.

- `LinearRegression`
- `DecisionTreeRegressor`
- `RandomForestRegressor`

각 모델의 내부 원리는 뒤의 장에서 자세히 다룬다.
여기서는 **서로 다른 모델을 어떻게 비교하고 선택하는가**에 집중한다.

### 모델 성능 평가

#### RMSE로 훈련 성능 확인

회귀 모델의 예측 오차는 **RMSE**(root mean squared error)라 불리는
평균 제곱근 오차로 평가할 수 있다.

RMSE는 예측 오차의 제곱의 평균값에 제곱근을 취한 값이며,
0에 가까울수록 실제값과 예측값의 차이가 작다는 뜻이다.

캘리포니아 주택가격으로 훈련된 세 예측 모델의 훈련셋에 대한 RMSE는 대략 다음과 같다.

| 모델 | 훈련셋 RMSE | 관찰 |
|---|---:|---|
| 선형 회귀 | 약 68,688 | 훈련 데이터에서도 오차가 큼 |
| 결정트리 | 0 | 훈련 데이터에는 완벽하게 맞음 |
| 랜덤 포레스트 | 약 17,474 | 선형 회귀보다 훈련 오차가 작음 |

하지만 **훈련셋 성능만으로는 어떤 모델이 새로운 데이터에서 잘 작동할지 판단할 수 없다.**
특히 결정트리의 RMSE가 0이라는 사실은 오히려 과대적합을 의심하게 한다.

#### 교차 검증으로 모델 비교하기

테스트셋은 최종 평가를 위해 남겨 두어야 한다.
그렇다면 훈련 과정에서 여러 모델을 어떻게 비교할 수 있을까?

**교차 검증**(cross-validation)은 훈련셋을 다시 여러 부분으로 나누어 모델을 반복해서 훈련하고 검증하는 방법이다.

k-겹 교차 검증에서는

1. 훈련셋을 k개의 **폴드**(fold)로 나눈다.
2. 하나의 폴드를 검증용으로 남기고 나머지로 모델을 훈련한다.
3. 검증용 폴드에서 성능을 측정한다.
4. 검증용 폴드를 바꾸어 k번 반복한다.
5. 여러 검증 결과를 함께 보고 모델의 성능을 판단한다.

아래 그림은 5-겹 교차 검증을 묘사한다.

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch02/cross-val10.png?raw=true" width="400">
</div>

교차 검증을 이용하면 한 번의 훈련 결과만 보는 것보다 모델의 일반화 성능을 더 안정적으로 비교할 수 있다.

아래 표는 5-겹 교차 검증을 통해 얻은 세 가지 모델의 성능(평균 RMSE)을 요약한다.

| 모델명 | 교차 검증 평균 RMSE | 성능 평가 요약 |
| :--- | :--- | :--- |
| **선형 회귀 모델** | 약 69,841 | 세 모델 중 가장 높은(나쁜) 오차를 보임 |
| **결정트리 회귀 모델** | 약 66,252 | 선형 회귀 모델보다 약간 낫지만 여전히 꽤 높은 오차를 보임 |
| **랜덤 포레스트 회귀 모델** | 약 47,224 | 훈련이 다소 오래 걸리지만 세 모델 중 성능이 가장 뛰어남 |

이 장에서 중요한 구분은 다음과 같다.

- **훈련셋: 모델 훈련과 모델 선택**  
- **교차 검증: 훈련셋 내부에서 모델 비교**  
- **테스트셋: 최종 모델의 마지막 평가**

### 하이퍼파라미터 미세 조정

선택된 모델을 이용하여 훈련을 시작하기 전에 사람이 
모델의 훈련과정동안 파라미터를 학습하는 알고리즘 세팅을 위해 사람이 정하는 설정값을 
모델의 **하이퍼파라미터**(hyperparameter)라고 한다.

예를 들어 랜덤 포레스트에서는 트리의 수나 일부 분할 관련 설정이 하이퍼파라미터다.

참고로 선택된 모델의 훈련을 통해 데이터에서 결정되는 값은 **모델 파라미터**(model parameter)이며
모델의 하이퍼파라미터와는 완전히 다른 값이다.

교차 검증 성능이 좋은 하이퍼파라미터 조합을 탐색하는 대표적인 방법은 다음과 같다.

- **그리드 탐색**(grid search): 미리 지정한 조합을 체계적으로 확인
- **랜덤 탐색**(randomized search): 탐색 공간에서 일부 조합을 무작위로 선택

### 최종 모델 평가

가장 좋은 성능을 내는 하이퍼파라미터로 모델의 훈련을 마친 뒤에야 테스트셋을 사용한다.

최종 모델에 테스트셋의 입력 데이터를 넣어 예측값을 만들고, 실제 타깃과 비교하여 RMSE를 계산한다.

이 값은 지금까지 모델 선택에 사용하지 않았던 데이터에서 측정한 결과이므로 **일반화 성능의 최종 추정치**로 사용한다.

테스트셋의 결과가 기대보다 좋지 않더라도, 같은 테스트셋을 반복해서 확인하며 모델을 다시 조정하면 최종 평가의 의미가 약해진다.

## 회귀 모델은 무엇을 학습하는가

**파라미터 → 손실/MSE → 경사하강법 직관 → 모델 복잡도 → 과소·과대적합 → 규제**

### 선형 회귀 

선형 회귀 모델을 이용하여 머신러닝 모델의 기능과
훈련 과정의 이해에 중요한 기초 개념을 살펴 본다.
선형 회귀 모델을 예제로 사용하는 이유는 크게 두 가지다.

첫째, 선형 회귀 모델의 훈련 과정이 매우 단순하여 머신러닝의 기초 개념을 설명하는 데에 매우 유용하다.

둘째, 딥러닝 심층 신경망 모델 등 대다수의 머신러닝 모델이 훈련 과정에서 선형 회귀 모델의 훈련 방식을 활용한다.

먼저 1장에서 1인당 GDP와 삶의 만족도 사이의 선형 관계를 학습한 선형 회귀 모델이 예측값을 계산하는 방식을 재확인한다.

**예제: 1인당 GDP와 삶의 만족도**

1인당 GDP와 삶의 만족도 사이의 관계를
다음 1차 방정식 함수로 표현하면 다음과 같다.

$$\text{삶의만족도} = \theta_0 + (\text{1인당GDP}) \cdot \theta_1$$

선형 회귀 모델은 1인당 GDP가 주어지면 삶의 만족도를 예측하는 모델을 훈련시키기 위해
위 함수를 활용한다. 

이를 보다 수학적으로 표현하면 다음과 같다. 

$$\hat y = \theta_0 + x_1 \cdot \theta_1$$

위 식에서 $x_1$은 1인당 GDP를,
$\hat y$는 예측된 삶의 만족도를 가리킨다.

**예제: 캘리포니아 주택 가격 예측**

2장에서 활용한 캘리포니아 주택 가격을 예측하는 선형 회귀 모델은
구역별로 주어진 9개의 특성값을 24개로 변환한 다음에 그 지역의 중위 주택 가격을 예측한다.
즉, 1개의 편향과 함께 24개의 가중치를 활용한 아래 모양의 함수를 이용하여 예측값을 계산한다.

$$\hat y = \theta_0 + x_1 \cdot \theta_1 + \cdots + x_{24} \cdot \theta_{24}$$

* $\hat y$: 구역의 예측된 중위 주택 가격
* $x_i$: 구역의 $i$ 번째 특성값(위도, 경도, 중간소득, 가구당 인원 등)
* $\theta_0$: 편향
* $\theta_i$: $i$ 번째 특성에 대한 가중치.

:::{note} 기울기 vs. 가중치

특성이 하나일 때는 기울기 표현이 적절했지만 특성이 2개 이상일 때는 더 이상 적절하지 않으며,
대신 각 특성값에 가해지는 **가중치**<font size='2'>weight</font>라는 표현이 선호된다.
:::

**선형 회귀 예측값**

위 두 예제에서 설명한 선형 회귀 모델이 
예측값을 생성할 때 사용하는 선형 함수를 일반화하면 다음과 같다.

먼저 훈련셋에 포함된 샘플이 $n$ 개의 특성 $x_1$, $x_2$, ..., $x_n$을 갖는다고 가정할 때,
선형 회귀 모델은 아래 식을 이용하여 예측값을 계산한다.
$\theta_0$와 1을 곱해주는 이유는 다른 항과의 형식을 맞추기 위함이다.

$$\hat y = 1\cdot \theta_0 + x_1 \cdot \theta_1 + \cdots + x_n \cdot \theta_{n}$$

아래 이미지는 위 식을 시각화한다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/perceptron02.png" width="250"/>
</div>

**선형 회귀 모델의 파라미터: 편향과 가중치**

머신러닝 모델을 훈련하는 주요 목표는 입력값과 타깃 사이에 존재하는 숨은 관계를 찾아내는 것이다.
예를 들어, 선형 회귀 모델은 입력값과 예측값 사이의 관계를 입력 특성과 가중치 $\theta_i$의 선형 조합을 통해 설명한다.
따라서 편향과 가중치는 훈련을 통해 학습해야 하는 파라미터이며, 이는 훈련 데이터셋으로부터 추론해야 하는 정보이다.
선형 회귀 모델을 훈련할 때는 보통 편향 $\theta_0$을 0으로, 나머지 가중치 $\theta_i$는 무작위 값으로 초기화한다.
이후 경사하강법을 적용하여 모든 파라미터(편향과 가중치)를 점차 더 적합한 값으로 업데이트한다.

#### 머신러닝 모델 훈련의 목표

머신러닝 모델의 훈련은 타깃에 최대한 가까운 예측값 계산을 목표로 한다.
예를 들어, 이전 예제에서 활용한 선형 회귀 모델은 좋은 예측값을 계산하기 위해 
적절한 5개의 파라미터 $\theta_0$, $\theta_1$, $\theta_2$, $\theta_3$, $\theta_4$를
훈련을 통해 학습해야 한다.

**비용 함수**

비용 함수<font size='2'>cost function</font>는 모델의 성능이 얼마나 나쁜가를 계산한다.
비용 함수가 계산하는 값이 작을 수록 해당 모델의 성능이 좋은 것이다.

:::{admonition} 비용 함수와 손실 함수
:class: note

비용 함수는 손실 함수<font size='2'>loss function</font>,
비용 함수에 의해 계산된 값은 손실값<font size='2'>loss</font>이라 부르기도 한다.
:::

**MSE: 회귀 모델의 비용 함수**

비용 함수는 모델 종류와 목표에 따라 다르게 정의되지만
회귀 모델의 경우 일반적으로 **평균 제곱 오차**<font size="2">mean squared error</font>(MSE)를
비용 함수로 사용한다.
선형 회귀 모델의 MSE를 계산하는 수식은 다음과 같다.

$$
\begin{align*}
\mathrm{MSE}(\mathbf{\theta})
&=
\frac{
(\mathbf{x}^{(1)} \mathbf{\theta} - y^{(1)})^2
+
(\mathbf{x}^{(2)} \mathbf{\theta} - y^{(2)})^2
+
\cdots
+
(\mathbf{x}^{(m)} \mathbf{\theta} - y^{(m)})^2
}{m}
\\[0.7ex]
&=
\frac{1}{m}
\sum_{i=1}^{m}
\left(
\mathbf{x}^{(i)} \mathbf{\theta} - y^{(i)}
\right)^2
\end{align*}
$$

위 수식에서 포함된 기호들의 의미는 다음과 같다.

| 기호 | 의미 |
| :---: | :--- |
| $\mathbf{x}^{(i)}$ | $i$ 번째 샘플. 단 0번 인덱스에 1이 추가됨. |
| $y^{(i)}$ | $i$ 번째 샘플에 대한 타깃 |
| $\mathbf{\theta}$ | 파라미터(편향과 가중치)로 구성된 1차원 어레이 $(\theta_0, \theta_1, \dots, \theta_n)$ |
| $m$ | 입력 데이터셋 크기 |

또한 $\mathbf{x}^{(i)}\, \mathbf{\theta}$ 는 $i$-번째 샘플에 대한 예측값 $\hat y^{(i)}$를 가리킨다.
이때 $i$는 $1$부터 $m$까지 움직인다.

$$
\hat y^{(i)} = \mathbf{x}^{(i)}\, \mathbf{\theta} = 1\cdot \theta_0 + x^{(i)}_1 \cdot \theta_1 + \cdots + x^{(i)}_n \cdot \theta_{n}
$$

**모델 훈련의 최종 목표**

회귀 모델의 경우 훈련셋이 주어졌을 때 $\mathrm{MSE}(\mathbf{\theta})$가 최소가 되도록 하는 
$\mathbf{\theta}$를 찾아야 한다.
선형 회귀의 경우 모델에 따라 다음 두 가지 방식 중 하나를 이용하여 해결한다.

* 방식 1: 정규방정식 또는 특이값 분해(SVD)
* 방식 2: 경사하강법(Gradient descent)

정규 방정식은 `LinearRegression` 등 선형 회귀를 활용하는 극히 일부 모델에서, 그것도 훈련셋의 크기와 입력 특성 개수가 모두 작을 때만 활용된다. 
반면에 `SGDRegressor` 등의 모델이 사용하는
경사하강법은 딥러닝 모델에서도 기본으로 활용되는 훈련 기법이다.
이런 의미에서 정규 방정식은 여기서는 다루지 않는다.

(sec:gradient-descent)=
### 경사하강법

**하이퍼파라미터<font size="2">hyperparameter</font>**

훈련시킬 모델을 지정할 때 사용되는 설정 옵션, 
즉 해당 클래스의 객체를 생성할 때 클래스의 생성자 함수에 전달되는 인자들을 가리킨다.
대표적으로 학습률, 에포크, 허용 오차, 배치 크기 등이 있다.

**파라미터**<font size="2">parameter</font>

선형 회귀 모델에 사용되는 편향과 가중치 파라미터처럼 모델 훈련중에 학습되는 값들을 가리킨다.
모델 훈련을 통해 학습된 파라미터는 훈련된 모델 객체의 속성으로 저장된다.

**비용 함수**

평균 제곱 오차(MSE)처럼 모델의 성능이 얼마나 나쁜가를 계산하는 함수다.
비용 함수의 값, 즉 손실값은 배치 단위로 계산된다.

:::{prf:example}

회귀 모델의 배치 단위로 계산되는 손실값은 일반적으로 평균 제곱 오차(MSE)로 계산된다.
예를 들어
`SGDRegressor` 모델은 크기가 1인 배치로 훈련하기에
매 스텝에서 하나의 입력 샘플 $\mathbf{x}^{(i)}$에 대해 MSE를 계산한다.

$$
\mathrm{MSE}(\mathbf{\theta}) = 
\big(\mathbf{x}^{(i)}\, \mathbf{\theta} - y^{(i)}\big)^2
$$
:::

**스텝**<font size='2'>step</font>

스텝은 하나의 배치에 대해 예측값과 손실값(비용)을 계산하고,
이후 손실값을 줄이는 방향으로 파라미터를 한 번 업데이트하는 과정이다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/step.png" width="400"/>
</div>

:::{prf:example}

`SGDRegressor` 모델은 크기가 1인 배치로 훈련하기에 하나의 스텝에 하나의 데이터 샘플만 활용한다.
즉, 아래 과정으로 구성된 스텝을 훈련셋에 포함된 각각의 샘플에 대해 반복해서 진행한다.

- 하나의 데이터 샘플에 대한 예측값 계산
- MSE 계산
- MSE의 그레이디언트 벡터 계산
- $\theta$ 업데이트
:::

**학습률($\eta$)**

훈련 스텝마다 파라미터 $\mathbf{\theta}$를 얼만큼씩 조정할 것인지를 정하는 비율이다.
학습률이 너무 크거나 작으면 모델이 적절한 파라미터를 제대로 학습하지 못할 수 있다.

**최적의 모델**

최종적으로 훈련을 통해 얻고자 하는 모델이며,
비용 함수의 값, 즉 손실값을 최소화하는 파라미터를 학습한(찾아낸) 모델이다.

#### 선형 회귀 모델의 경사하강법

MSE를 비용 함수로 사용하는 선형 회귀 모델의 파라미터를 업데이트하는 스텝에서
사용되는 경사하강법은 아래 과정으로 구성된다.

1. 파라미터 벡터 $\mathbf{\theta}$를 0 또는 임의의 값으로 초기화한 후에 훈련을 시작한다.

1. 지정된 에포크만큼 또는 그레이디언트 벡터
    $\nabla_\mathbf{\theta} \textrm{MSE}(\mathbf{\theta})$가 충분히 작아질 때까지 
    아래 과정으로 구성된 훈련 스텝을 반복한다.

    * 하나의 배치에 대해 예측값 생성 후 손실값 $\mathrm{MSE}(\mathbf{\theta})$ 계산.
    * $\mathbf{\theta}$를 아래 점화식을 이용하여 업데이트:

    $$
    \theta^{(\text{new})} = \theta^{(\text{old})}\, -\, \eta\cdot \nabla_\theta \textrm{MSE}(\theta^{(\text{old})})
    $$

    위 식에서 $\eta$는 학습률, 
    $\theta^{(\text{old})}$는 이전 스텝을 통해 얻어진 파라미터 벡터, 
    $\theta^{(\text{new})}$는 업데이트된 파라미터 벡터를 가리킨다.

:::{note} 그레이디언트 벡터의 방향과 크기

그레이디언트 벡터 $\nabla_\mathbf{\theta} \textrm{MSE}(\mathbf{\theta})$가 가리키는 방향의
**반대 방향**으로 움직이면 빠르게 $\textrm{MSE}(\mathbf{\theta})$가 
전역 최소값을 갖는 $\mathbf{\theta}$ 접근한다.

아래 이미지는 선형회귀 모델의 손실 함수에 대한 가중치 업데이트를 진행하는 과정을 보여준다.
이해를 위해 먼저 파라미터 벡터 $\mathbf{\theta}$를 벡터가 아닌 하나의 가중치 값으로 간주하면
MSE가 $\mathbf{\theta}$에 대한 2차 함수로 표현됨에 주의한다.

따라서 비용 함수 그래프상의 한 점에서의 기울기는 비용 함수의 $\mathbf{\theta}$에 대한 미분값으로
계산되는 데 이 값이 바로 그레이디언트 벡터에 해당한다. 
그리고 기울기가 양수(또는 음수)이면 기존의 가중치(weight)를 왼쪽(또는 오른쪽)으로 움직여야 비용 함수의 전역 최소값에 가까워진다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/gradient01b.png" width="500"/>
</div>

<p><div style="text-align: center">&lt;이미지 출처: <a href="https://www.analyticsvidhya.com/blog/2020/10/how-does-the-gradient-descent-algorithm-work-in-machine-learning/">Analytics Vidhya</a>&gt;</div></p>

아래 두 이미지는 산에서 가장 경사가 급한 길을 따를 때 가장 빠르게 하산한다는 원리를 보여준다.
이유는 해당 지점에서 그레이디언트 벡터를 계산하면 정상으로 가는 가장 빠른 길을 안내하기에
그 반대방향으로 움직여야 최대한 빠르게 최저점(평지)으로 하산할 수 있기 때문이다.
이미지에서 보여지는 여러 경로는 경사하강법 알고리즘에 따른 다른 경로를 보여준다.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/saddle_point_evaluation_optimizers.gif" width="500"/>
</div>
:::

#### 학습률의 중요성

(sec:poly_reg)=
### 비선형 데이터 학습: 다항 회귀

**다항 회귀**<font size="2">polynomial regression</font>는
비선형 데이터를 선형 회귀를 이용하여 학습하는 기법이다.

**예제: 2차 함수 모델를 따르는 데이터셋에 선형 회귀 모델 적용**

아래 이미지는 2차 함수의 그래프 형식으로 분포된 데이터셋을 선형 회귀 모델로 학습시킨 결과를 보여준다.
즉, 예측값이 $x_1$에 대한 아래 1차 함수로 계산된다.

$$\hat y = \theta_0 + \theta_1\, x_1$$

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-06.png" width="500"/>
</div>

**예제: 2차 함수 모델를 따르는 데이터셋에 2차 다항 회귀 모델 적용**

반면에 아래 이미지는 $x_1^2$ 에 해당하는 특성을 새로이 추가한 후에
선형 회귀 모델을 학습시킨 결과를 보여준다.
예측값은 $x_1$에 대한 아래 2차 함수 형식으로 계산된다.

$$\hat y = \theta_0 + \theta_1\, x_1 + \theta_2\, x_{1}^2$$

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-07.png" width="500"/>
</div>

**다항 회귀의 단점**

몇차 다항 회귀를 사용해야 할지 일반적으로 알 수 없다. 
또한 심층 신경망처럼 비선형 데이터를 분석하는 보다 좋은 모델이 개발되어 굳이 다항 회귀를 사용할 필요가 없어졌다.
여기서는 비선형 데이터 분석을 선형 회귀 모델로 제대로 예측할 수 없음을 보여주기 위해 언급되었다.

### 과소/과대 적합

사용되는 모델에 따라 훈련된 모델의 성능이 많이 다를 수 있다.
아래 이미지는 기본 선형 모델은 성능이 너무 좋지 않은 반면에
300차 다항 회귀 모델은 너무 과하게 훈련 데이터에 민감하게 반응하는 것을 보여준다.
반면에 2차 다항 회귀 모델이 적절(?)하게 예측값을 계산하는 것으로 보인다.

<div align="center"><img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/homl04-08.png" width="500"/></div>

**모델 성능 평가: 교차 검증**

일반적으로 어떤 모델이 가장 좋은지 미리 알 수 없다. 
따라서 보통 다양한 모델을 대상으로 교차 검증을 진행하여 성능을 평가한다.
교차 검증 결과에 따른 모델 평가는 다음 두 종류로 나뉜다.

* 과소적합: 훈련 점수와 교차 검증 점수 모두 낮은 경우
* 과대적합: 훈련 점수는 높지만 교차 검증 점수가 상대적으로 많이 낮은 경우

**과소 적합 모델 개선법**

모델이 데이터셋에 과소 적합 하는 일반적인 경우와 개선법은 다음과 같다.

- 너무 단순한 모델 활용: 보다 복잡한 데이터를 학습할 수 있는 모델 활용
- 너무 적은 데이터: 훈련셋 양 늘리기

**과대 적합 모델 개선법**

- 모델 규제 적용: 모델이 훈련중에 데이터에 너무 민감하게 반응하지 않도록 규제 가함. 이어지는 내용 참고
- 보다 많은 데이터: 데이터가 많을 수록 일반적으로 과대 적합이 덜 발생함.

**모델의 일반화 성능**

훈련 과정에서 다루지 않은 새로운 데이터 대한 예측 능력이 모델의 **일반화 성능**이다.
새로운 데이터에 대한 모델의 예측에 나쁜 영향을 미치는 요소는 일반적으로 다음 세 가지가 있다.

- 편향: 실제로는 2차원 모델인데 1차원 모델을 사용하는 경우처럼 데이터의 분포에 대한 잘못된 가정으로 인해 발생한다.
    과소 적합이 발생할 가능성이 매우 높다.

- 분산: 모델이 훈련 데이터에 민감하게 반응하는 정도를 가리킨다.
    고차 다항 회귀 모델처럼 모델이 학습해야하는 파라미터의 수가 많을 수록 분산이 커진다.
    
- 제거 불가능 오류: 노이즈(noise) 등 데이터 자체의 한계로 인해 발생한다.
    데이터 전처리 과정에서 노이즈를 제거해야만 오류를 줄일 수 있다.

**모델 자유도**

모델이 학습해야 하는 파라미터의 수를 모델의 **자유도**<font size='2'>degree of freedom</font>라 부르기도 한다.
자유도가 높을 수록 복잡한 방식으로 예측값을 계산하고, 일반적으로 분산이 크다.

**편향-분산 트레이드오프**

복잡한 모델일 수록 편향을 줄어들지만 분산은 커지는 현상을 가리킨다.

### 모델 규제

훈련 중에 과소 적합이 발생하면 보다 복잡한 모델을 선택해야 한다.
반면에 과대 적합이 발생할 경우 보다 단순한 모델을 사용하거나 모델에 규제를 가해서
모델의 분산을 줄여 과대 적합을 방지하거나 과대 적합이 최대한 늦게 발생하도록 유도해야 한다. 

회귀 모델에 대한 **규제**<font size='2'>regularization</font>는 가중치의 역할을 제한하는 방식으로 이루어지며,
방식에 따라 다음 세 가지 회귀 모델이 지정된다.

* 릿지 회귀
* 라쏘 회귀
* 엘라스틱 넷

**릿지 회귀<font size='2'>Ridge Regression</font>**

아래 비용 함수를 사용한다.

$$J(\theta) = \textrm{MSE}(\theta) + \frac{\alpha}{m} \sum_{i=1}^{n}\theta_i^2$$

* $\theta_0$: 규제에서 제외.
* $m$: 배치 크기
* $\alpha$(알파): 규제 강도. $\alpha=0$일 때 규제 없음.
    - $\alpha$ 가 커질 수록 가중치의 역할이 줄어듦.
        비용을 줄이기 위해 가중치 $\theta_i$를 작게 유지하도록 훈련되어 결국 모델의 분산 정도가 작아짐.

:::{tip}

`StandardScaler` 등을 사용하여 특성 스케일링을 진행 한 다음에
규제를 적용해야 모델의 성능이 좋아진다.
이유는 $\theta_i$가 특성의 크기에 의존하기에
모든 특성의 크기를 비슷하게 맞추면 $\theta_i$가 
보다 일정하게 수렴하기 때문이다.
:::

아래 이미지는 서로 다른 규제 강도를 사용한 릿지 회귀 모델의 훈련 결과를 보여준다.

- 왼쪽: 선형 회귀 모델에 세 개의 $\alpha$ 값 적용.
- 오른쪽: 10차 다항 회귀 모델에 세 개의 $\alpha$ 값 적용.

<div align="center">
    <img src="https://raw.githubusercontent.com/codingalzi/code-workout-ml/master/images/ch04/ridge01.png" width="600"/>
</div>

## P1 이론 정리

> **문제 정의 → 데이터 이해와 분리 → EDA와 특성 검토 → 전처리와 Pipeline → 모델 훈련과 평가 → 검증 → 모델 학습 원리 → 일반화와 규제 → 최종 테스트**